In [0]:
from pyspark.sql import functions as F

bronze = spark.table("oece.bronze.seace_records")
v = "compiled_release.tender.value"

df_tc = (
    bronze
    .select(
        "periodo",
        F.col(f"{v}.currency").alias("moneda"),
        F.col(f"{v}.amount").alias("monto"),
        F.col(f"{v}.amount_PEN").alias("monto_pen"),
    )
    .filter("moneda <> 'PEN' AND monto > 0 AND monto_pen > 0")
    .groupBy("moneda", "periodo")
    .agg(
        F.round(F.expr("percentile_approx(monto_pen / monto, 0.5)"), 4).alias("tipo_cambio"),
        F.count("*").alias("observaciones"),
    )
)

df_tc.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("oece.silver.tipo_cambio")

display(spark.table("oece.silver.tipo_cambio").orderBy("moneda", "periodo"))

moneda,periodo,tipo_cambio,observaciones
EUR,2026-08,4.054,5
EUR,2026-09,3.925,4
EUR,2026-10,3.893,5
EUR,2026-11,3.874,3
GBP,2026-08,4.659,2
USD,2026-08,3.402,55
USD,2026-09,3.41,53
USD,2026-10,3.448,62
USD,2026-11,3.466,11
